In [1]:
import pandas as pd
import numpy as np 
import re 
import pickle 
import tensorflow as tf 

In [2]:
df=pd.read_csv('english_urdu_dataset.csv')

In [3]:
df.isnull().sum()

english    0
urdu       0
dtype: int64

In [4]:
df['english']=df['english'].astype(str)
df['urdu']=df['urdu'].astype(str)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   english  10000 non-null  str  
 1   urdu     10000 non-null  str  
dtypes: str(2)
memory usage: 887.1 KB


In [6]:
df['english']=df['english'].str.lower()

In [7]:
df['english']=df['english'].str.strip()
df['urdu']=df['urdu'].str.strip()

In [8]:
df["urdu"] = "<sos> " + df["urdu"] + " <eos>"

In [9]:
df

,english,urdu
0,i eat apples today.,<sos> میں سیب آج کھاتا ہوں۔ <eos>
1,i eat apples tomorrow.,<sos> میں سیب کل کھاتا ہوں۔ <eos>
2,i eat apples every day.,<sos> میں سیب ہر روز کھاتا ہوں۔ <eos>
3,i eat apples now.,<sos> میں سیب ابھی کھاتا ہوں۔ <eos>
4,i eat apples at home.,<sos> میں سیب گھر پر کھاتا ہوں۔ <eos>
...,...,...
9995,you watch dogs today. #6996,<sos> تم کتے آج دیکھتا ہوں۔ #6996 <eos>
9996,you watch dogs tomorrow. #6997,<sos> تم کتے کل دیکھتا ہوں۔ #6997 <eos>
9997,you watch dogs every day. #6998,<sos> تم کتے ہر روز دیکھتا ہوں۔ #6998 <eos>
9998,you watch dogs now. #6999,<sos> تم کتے ابھی دیکھتا ہوں۔ #6999 <eos>


In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer_eng=Tokenizer(filters="")
tokenizer_urd=Tokenizer(filters="")

In [11]:
tokenizer_eng.fit_on_texts(df['english'])
tokenizer_urd.fit_on_texts(df['urdu'])

In [12]:
len(tokenizer_eng.word_index)
len(tokenizer_urd.word_index)

7034

In [13]:
seq_eng=tokenizer_eng.texts_to_sequences(df['english'])
seq_urd=tokenizer_urd.texts_to_sequences(df['urdu'])

In [14]:
with open("tokenizer_eng.pkl", 'wb')as file:
    pickle.dump(tokenizer_eng, file)
with open("tokenizer_urd.pkl", 'wb')as file:
    pickle.dump(tokenizer_urd, file)    

In [15]:
max_len=1000

In [16]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
encoder_seq=pad_sequences(
    seq_eng, 
    maxlen=max_len,
    padding='pre'
)
decoder_seq=pad_sequences(
    seq_urd,
    maxlen=max_len,
    padding='pre'
)


In [17]:
decoder_input = decoder_seq[:, :-1]
decoder_output = decoder_seq[:, 1:]

In [18]:
decoder_output.shape

(10000, 999)

In [19]:
from sklearn.model_selection import train_test_split
enc_train, enc_test, dec_inp_train, enc_inp_test, dec_out_train,dec_out_test=train_test_split(
    encoder_seq,
    decoder_input,
    decoder_output, test_size=0.2, random_state=42
)

In [20]:
print(    decoder_input.shape)

(10000, 999)


In [21]:
import tensorflow as tf

In [22]:
eng_vocab = len(tokenizer_eng.word_index) + 1

urdu_vocab = len(tokenizer_urd.word_index) + 1



In [23]:
encoder_input=tf.keras.layers.Input(shape=(1000, ))
x=tf.keras.layers.Embedding(input_dim=eng_vocab, output_dim=128)(encoder_input)
_,h,c=tf.keras.layers.LSTM(
    256, return_state=True
)(x)

In [24]:
decoder_input=tf.keras.layers.Input(shape=(999, ))
y=tf.keras.layers.Embedding(input_dim=urdu_vocab, output_dim=128)(decoder_input)
decoder_output=tf.keras.layers.LSTM(
    256,
    return_sequences=True,

    return_state=True
)
decoder_output, _, _ = decoder_output(
    y,
    initial_state=[h, c]
)

In [25]:
output=tf.keras.layers.Dense(urdu_vocab, activation='softmax')(decoder_output)
model=tf.keras.Model([encoder_input, decoder_input], output)

In [26]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

In [27]:
print(model.output_shape)

print(model.summary())

print(urdu_vocab)

(None, 999, 7035)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 1000)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 999)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1000, 128) │    900,352 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 999, 128)  │    900,480 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    394,240 │ embedding[0][0]   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, 999,      │    394,240 │ embedding_1[0][0… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 999, 7035) │  1,807,995 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,397,307 (16.77 MB)

 Trainable params: 4,397,307 (16.77 MB)

 Non-trainable params: 0 (0.00 B)

None
7035


In [29]:
model.fit(
    [enc_train,dec_inp_train],
    dec_out_train,
    epochs=1,
    verbose=1
)

250/250 ━━━━━━━━━━━━━━━━━━━━ 2646s 11s/step - loss: 0.0277


In [30]:
model.save('encoder_decoder.h5')

In [31]:
print(enc_train.shape)
print(dec_inp_train.shape)
print(dec_out_train.shape)

(8000, 1000)
(8000, 999)
(8000, 999)
